# RetractorDB as a data source, from a notebook

J1-J3 of the embedded programme: a plan compiled and executed in-process, one time
slot at a time, with its streams readable as records, NumPy arrays and DLPack
windows. No daemon, no shared memory, no `xqry`. See `docs/jupyter-integration.md`
section 7 and `docs/core-phase-3.md`.

**Prerequisite.** The tree must be configured with `-DRDB_PYTHON=ON` and built. The
cell below finds the module in the build tree; nothing needs to be installed.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[2]  # api/python/notebooks -> repo root

roots = [p for p in sorted((REPO / 'build').glob('*/python')) if any((p / 'retractordb').glob('_core*'))]
assert roots, 'no built extension found - configure with -DRDB_PYTHON=ON and build'
sys.path.insert(0, str(roots[0]))

import retractordb as rdb
print('module root:', roots[0])


## A plan

The same plan as `test/IntegrationTest/untileof_stop`: a text source sampled every
half second, and one SELECT doubling it. The engine gets a private directory for the
plan's storage files; `FILE` paths are resolved by the process, so they are absolute
here.


In [ ]:
import tempfile

workdir = Path(tempfile.mkdtemp())
(workdir / 'data.txt').write_text(''.join(f'{v}\n' for v in (10, 20, 30, 40, 50, 60, 70, 80)))

PLAN = f"""
DECLARE a INTEGER STREAM src, 1/2 FILE '{workdir / 'data.txt'}'
SELECT a*2 STREAM dst FROM src
"""

eng = rdb.Engine(str(workdir))
eng.compile(PLAN)
print(eng.streams())
print(eng.schema('dst'))


## Stepping

`step()` advances one time slot and returns its index. With `until_eof=True` (the
default) a declared source is read once, without wrapping past the end of its file,
and `step()` returns `None` after the slot in which it ran dry. `run()` does the same
in a loop with the GIL released - the stop button works, because `run()` checks for
`KeyboardInterrupt` every 50 ms.


In [ ]:
print('first slots:', eng.step(), eng.step(), eng.step())
print('remaining slots run:', eng.run())
print('slots done:', eng.slots_done, ' plan time:', eng.time, ' end of input:', eng.end_of_input)
print('dst:', [row[0] for row in eng.rows('dst')])


## Arrays and windows

`to_numpy()` is a dense `(records, values)` block of the fields you name (all data
fields by default), NULL as NaN. `window()` stacks windows of `size` records `stride`
apart, shape `(n_windows, size, n_values)` - what a training batch is made of.
Everything is a copy of engine memory, so a window is still valid after the engine
moves on.


In [ ]:
block = eng.to_numpy('dst')
print(block.shape, block.dtype)
print(block[:, 0])

w = eng.window('dst', fields=['dst_0'], size=4, stride=2, dtype='float32')
print(w)
print(w[0][:, 0], w[-1][:, 0])


## Into PyTorch, without depending on it

`Window` exports the DLPack protocol, so `torch.from_dlpack(w)` shares the window's
memory with a tensor - one copy from the engine, none after. The package never
imports torch itself; `retractordb.torch` wraps the same call in an
`IterableDataset` for a `DataLoader` (with `num_workers=0`: the engine does not
survive `fork`).


In [ ]:
try:
    import torch
except ImportError:
    print('torch is not installed; numpy.from_dlpack shows the same protocol at work')
    import numpy
    print(numpy.from_dlpack(w).shape)
else:
    t = torch.from_dlpack(w)
    print(t.shape, t.dtype)
    from retractordb.torch import StreamDataset
    ds = StreamDataset(eng, 'dst', fields=['dst_0'], window=3, stride=1)
    print(len(ds), next(iter(ds)).shape)


## What raises

A plan that does not parse raises `RQLSyntaxError`; one that does not compile, or
uses something the embedded engine does not have (`DUMP` / `SYSTEM` rules,
`ROTATION`), raises `CompileError`. Both are `ConfigError`: your input, intact engine,
live kernel. Engine diagnostics go to `logging.getLogger('retractordb')`, not to the
cell's output.


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

for attempt in (lambda: eng.compile('SELEKT nonsense'),
                lambda: eng.compile('SELECT a STREAM dst FROM nosuch'),
                lambda: eng.compile(PLAN + "RULE r ON dst WHEN dst[0] > 1 DO SYSTEM 'echo no'"),
                lambda: rdb.Engine(str(workdir / 'nowhere')).compile(PLAN)):
    try:
        attempt()
    except rdb.ConfigError as exc:
        print(type(exc).__name__, '-', str(exc).splitlines()[0])

eng.close()
